# Import the packages and set the hyperparameters



In [1]:
### it's important to choose how we get to have the labelled data from 25600 datapoints. Are they randomly selected?
### Are they from lhs? Are they evenly disributed?

import tensorflow as tf
import datetime, os
#hide tf logs 
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'} 
#0 (default) shows all, 1 to filter out INFO logs, 2 to additionally filter out WARNING logs, and 3 to additionally filter out ERROR logs
import scipy.optimize
import scipy.io
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.axes_grid1 import make_axes_locatable
from mpl_toolkits.mplot3d import Axes3D
import time
from pyDOE import lhs         #Latin Hypercube Sampling
import seaborn as sns 
import codecs, json

# hyperparameters:
foldername = 'run6_minus_initial' # creates a folder with this name in the current path where the results will be saved.
seed = 3
np.random.seed(seed)
tf.random.set_seed(seed)
maxiter = 500
N_b = 100 # no. of training data on the boundary
N_f = 10000 # no. of training data inside the domain
N_l = 1000 # no. of labelled data for training 
layers = np.array([2,15,15,15,15,1])
initial_lambda2 = -3.0


# Data Prep

Training and Testing data is prepared from the solution file

In [2]:
newFolder = os.path.join(os.getcwd(), foldername)
if not os.path.exists(newFolder): os.makedirs(newFolder)

# data = scipy.io.loadmat('Data/burgers_shock.mat')   
# data = scipy.io.loadmat('Data/burgers_shock_mu_005_pi.mat') # lambda2 = 0.0015915495
data = scipy.io.loadmat('Data/burgers_shock_IC_sin2pi.mat') # # lambda2 = 0.0031830989 
# data = scipy.io.loadmat('Data/burgers_shock_mu_01_pi.mat')  # lambda2 = 0.0031830989
x = data['x']                                   # 256 points between -1 and 1 [256x1]
t = data['t']                                   # 100 time points between 0 and 1 [100x1] 
usol = data['usol']                             # solution of 256x100 grid points

X, T = np.meshgrid(x, t)                        # makes 2 arrays X and T such that u(X[i],T[j])=usol[i][j] are a tuple

# Test Data

Test data includes the entire dataset and helps with the comparison with the PINN predictions.

In [3]:
''' X_u_test = [X[i],T[i]] [25600,2] for interpolation'''
X_u_test = np.hstack((X.flatten()[:,None], T.flatten()[:,None]))
lb = X_u_test[0]  # [-1. 0.]
ub = X_u_test[-1] # [1.  0.99]
u = usol.flatten('F')[:,None] # column-major order

# Training Data

Training data contains 3 types of data:

1) **Labelled** data subset: N_l points inside the domain where the (x,t) coordinate and its u solution is assumed given. This subset is necessary in inverse problem PDEs.

2) **Boundary** data subset: N_b points on the boundary where the (x,t) coordinate and its u solution is assumed given.

2) **Residual** data subset: N_f points inside the domain where the (x,t) coordinate (but not its u solution) is assumed given.

In [4]:
# def trainingdata(N_b, N_f):
def trainingdata(N_b, N_f, N_l):

    '''labelled data inside the domain'''
    
    X_all_labelled = np.hstack((X.flatten()[:,None], T.flatten()[:,None])) # all labelled 25600 X pairs
    print(X_all_labelled)
    u_all_labelled = usol.flatten('F')[:,None] # all 25600 labelled u's
    idx_l = np.random.choice(X_all_labelled.shape[0], N_l, replace=False) # the index of laballed data
    X_labelled = X_all_labelled[idx_l, :] # choose (x,t) pairs from the set X_all_labelled with indices 'idx_l'
    u_labelled = u_all_labelled[idx_l, :] # choose u's from the set u_all_labelled with indices 'idx_l'

    '''the data that should meet the Boundary Conditions'''

    #Initial Condition -1 =< x =<1 and t = 0  
    leftedge_x = np.hstack((X[0,:][:,None], T[0,:][:,None])) #L1
    leftedge_u = usol[:,0][:,None]

    #Boundary Condition x = -1 and 0 =< t =<1
    bottomedge_x = np.hstack((X[:,0][:,None], T[:,0][:,None])) #L2
    bottomedge_u = usol[-1,:][:,None] # bottomedge_u = usol[0,:][:,None]

    #Boundary Condition x = 1 and 0 =< t =<1
    topedge_x = np.hstack((X[:,-1][:,None], T[:,0][:,None])) #L3
    topedge_u = usol[0,:][:,None] # topedge_u = usol[-1,:][:,None]

    all_X_u_train = np.vstack([leftedge_x, bottomedge_x, topedge_x]) # all_X_u_train [456,2] (456 = 256(L1)+100(L2)+100(L3))
    all_u_train = np.vstack([leftedge_u, bottomedge_u, topedge_u])   #and the corresponding u [456x1]

    #choose random N_b points for training
    idx_b = np.random.choice(all_X_u_train.shape[0], N_b, replace=False) # index of boundary points

    X_u_train = all_X_u_train[idx_b, :] #choose indices from  set 'idx_b' (x,t)
    u_train = all_u_train[idx_b,:]      #choose corresponding u

    '''Residual PDE Points'''

    # Latin Hypercube sampling for collocation points 
    # N_f sets of tuples(x,t)
    X_f_train = lb + (ub-lb)*lhs(2, N_f) 
    X_f_train = np.vstack((X_f_train, X_u_train)) # append training points to collocation points 

#     return X_f_train, X_u_train, u_train 
    return X_f_train, X_u_train, u_train, X_labelled, u_labelled


# **Inverse Problem PINN (Physics-Informed Neural Networks)** 

Generates a **PINN** of L hidden layers, each with n neurons. The functions are as follows:
1) **forward**: the NN (neural network) that predicts the output u, given the coordinates (x,t) as input
2) **get_weights**: transforms the NN trainable_variables (NN weights, NN biases, and the PDE parameters) from matrix form to 1D (one-dimensional), as required by the LBFGS optimizer in TensorFlow
3) **set_weights**: transforms the NN weights and biases from  1D to matrix form for making calculations easier.
4) **loss_labelled**: the loss function from the labelled data 
5) **loss_BC**: the loss function from the boundary and initial conditions
6) **loss_PDE**: the loss function from the PDE residuals (this term makes the PINNs unique to a simple function fitting)
7) **loss**: adds up all the 3 terms.
8) **optimizerfunc**: gets the 1D trainable_variables as input then returns the training loss and trainable_variables gradients. (this is the heart of the program)
9) **optimizer_callback**: this function is called after each iteration and makes the training records for later visualization.

In [5]:
class Sequentialmodel(tf.Module): 
    def __init__(self, layers, name=None):
       
        self.W = []  #Weights and biases
        self.parameters = 0 #total number of parameters
        self.iteration = 0
        self.losss_history = np.array([])
        self.error_history = np.array([])
        self.w_history = np.array([])
        self.lambda2_history = np.array([])
        self.lambda2 = tf.Variable([initial_lambda2], dtype='float64', trainable = True)
        self.parameters += 1
        
        for i in range(len(layers)-1):
            
            input_dim = layers[i]
            output_dim = layers[i+1]
            
            #Xavier standard deviation 
            std_dv = np.sqrt((2.0/(input_dim + output_dim)))

            #weights = normal distribution * Xavier standard deviation + 0
            w = tf.random.normal([input_dim, output_dim], dtype = 'float64') * std_dv                       
            w = tf.Variable(w, trainable=True, name = 'w' + str(i+1))
            b = tf.Variable(tf.cast(tf.zeros([output_dim]), dtype = 'float64'), trainable = True, name = 'b' + str(i+1))                    
            self.W.append(w)
            self.W.append(b)            
            self.parameters +=  input_dim * output_dim + output_dim
    
    def forward(self, x):
        
        x = (x-lb)/(ub-lb)        
        a = x
        for i in range(len(layers)-2):            
            z = tf.add(tf.matmul(a, self.W[2*i]), self.W[2*i+1])
            a = tf.nn.tanh(z)            
        a = tf.add(tf.matmul(a, self.W[-2]), self.W[-1]) # For regression, no activation to last layer
        print(a)
        print(self.W[2*i])
        return a
    
    def get_weights(self):

        parameters_1d = []  # [.... W_i,b_i.....  ] 1d array
        
        for i in range (len(layers)-1):
            
            w_1d = tf.reshape(self.W[2*i],[-1])   #flatten weights 
            b_1d = tf.reshape(self.W[2*i+1],[-1]) #flatten biases
            
            parameters_1d = tf.concat([parameters_1d, w_1d], 0) #concat weights 
            parameters_1d = tf.concat([parameters_1d, b_1d], 0) #concat biases
        parameters_1d = tf.concat([parameters_1d, self.lambda2], 0)
        return parameters_1d
        
    def set_weights(self,parameters):
                
        for i in range (len(layers)-1):

            shape_w = tf.shape(self.W[2*i]).numpy() # shape of the weight tensor
            size_w = tf.size(self.W[2*i]).numpy() #size of the weight tensor 
            
            shape_b = tf.shape(self.W[2*i+1]).numpy() # shape of the bias tensor
            size_b = tf.size(self.W[2*i+1]).numpy() #size of the bias tensor 
                        
            pick_w = parameters[0:size_w] #pick the weights 
            self.W[2*i].assign(tf.reshape(pick_w,shape_w)) # assign  
            parameters = np.delete(parameters,np.arange(size_w),0) #delete 
            
            pick_b = parameters[0:size_b] #pick the biases 
            self.W[2*i+1].assign(tf.reshape(pick_b,shape_b)) # assign 
            parameters = np.delete(parameters,np.arange(size_b),0) #delete 
            
        self.lambda2.assign(parameters[0:1]) # assign the PDE parameter   
        parameters = np.delete(parameters, np.arange(1), 0) #delete
    
    def loss_labelled(self, x, y):
        loss_l = tf.reduce_mean(tf.square(y-self.forward(x)))
        return loss_l
            
    def loss_BC(self, x, y):
        loss_u = tf.reduce_mean(tf.square(y-self.forward(x)))        
        return loss_u

    def loss_PDE(self, x_to_train_f):
    
        g = tf.Variable(x_to_train_f, dtype = 'float64', trainable = False)

        x_f = g[:,0:1]
        t_f = g[:,1:2]

        with tf.GradientTape(persistent=True) as tape:

            tape.watch(x_f)
            tape.watch(t_f)

            g = tf.stack([x_f[:,0], t_f[:,0]], axis=1)   
            z = self.forward(g)
            u_x = tape.gradient(z, x_f)

        u_t = tape.gradient(z, t_f)    
        u_xx = tape.gradient(u_x, x_f)

        del tape

        f = u_t + (self.forward(g))*(u_x) - self.lambda2*u_xx #### why not use z?

        loss_f = tf.reduce_mean(tf.square(f))

        return loss_f
    
    def loss(self, x, y, g, a, b):
        loss_l = self.loss_labelled(a, b)
        loss_u = self.loss_BC(x, y)
        loss_f = self.loss_PDE(g)

        loss = loss_u + loss_f + loss_l
        return loss, loss_u, loss_f, loss_l
    
    def optimizerfunc(self, parameters):
        
        self.set_weights(parameters)
       
        with tf.GradientTape() as tape:
            tape.watch(self.trainable_variables)
            loss_val, loss_u, loss_f, loss_l = self.loss(X_u_train, u_train, X_f_train, X_labelled, u_labelled)  
        grads = tape.gradient(loss_val, self.trainable_variables)      
        del tape
        
        grads_1d = [ ] #flatten grads 
        
        for i in range (len(layers)-1):

            grads_w_1d = tf.reshape(grads[2*i], [-1]) #flatten weights 
            grads_b_1d = tf.reshape(grads[2*i+1], [-1]) #flatten biases

            grads_1d = tf.concat([grads_1d, grads_w_1d], 0) #concat grad_weights 
            grads_1d = tf.concat([grads_1d, grads_b_1d], 0) #concat grad_biases
        grads_lambda2_1d = tf.reshape(grads[-1],[-1]) #flatten the gradient vector of PDE parameter (which is the last element) 
        grads_1d = tf.concat([grads_1d, grads_lambda2_1d], 0) #concat grads of lambda2

        return loss_val.numpy(), grads_1d.numpy()
    
    def optimizer_callback(self, parameters):
        self.iteration += 1
        
        loss_value, loss_u, loss_f, loss_l = self.loss(X_u_train, u_train, X_f_train, X_labelled, u_labelled)
        
        u_pred = self.forward(X_u_test)
        error = np.linalg.norm((u-u_pred),2)/np.linalg.norm(u,2)
        self.error_history = np.append(self.error_history, error)
        
        self.w_history = np.append(self.w_history, self.W[0][0][0]) ### putting the probe on a single weight amongst all.
        
        self.losss_history = np.append(self.losss_history, loss_value.numpy())
        print('iteration:', self.iteration, 'lambda2:', self.lambda2[0].numpy(), 'error:', error, 'loss_value:', loss_value.numpy())
        self.lambda2_history = np.append(self.lambda2_history, self.lambda2.numpy())

# *Solution Plot*

# *Model Training and Testing*

A function '**model**' is defined to generate a NN as per the input set of hyperparameters, which is then trained and tested. The L2 Norm of the solution error is returned as a comparison metric

In [ ]:
# Training data
X_f_train, X_u_train, u_train, X_labelled, u_labelled = trainingdata(N_b, N_f, N_l)

PINN = Sequentialmodel(layers)

init_params = PINN.get_weights().numpy()

start_time = time.time() 

results = scipy.optimize.minimize(fun = PINN.optimizerfunc, 
                                  x0 = init_params, 
                                  args=(), 
                                  method='L-BFGS-B', 
                                  jac= True,        # If jac is True, fun is assumed to return the gradient along with the objective function
                                  callback = PINN.optimizer_callback, 
                                  options = {'disp': None,
                                            'maxcor': 200, 
                                            'ftol': 1 * np.finfo(float).eps,  #The iteration stops when (f^k - f^{k+1})/max{|f^k|,|f^{k+1}|,1} <= ftol
                                            'gtol': 5e-8, 
                                            'maxfun':  50000, 
                                            'maxiter': maxiter,
                                            'iprint': -1,   #print update every 50 iterations
                                            'maxls': 50})

elapsed = time.time() - start_time              

[[-1.          0.        ]
 [-0.99215686  0.        ]
 [-0.98431373  0.        ]
 ...
 [ 0.98431373  0.99      ]
 [ 0.99215686  0.99      ]
 [ 1.          0.99      ]]
tf.Tensor(
[[ 1.49652532e-01]
 [ 4.90973425e-02]
 [ 1.64957085e-01]
 [ 3.29144429e-02]
 [ 1.72796405e-01]
 [ 7.91725058e-02]
 [ 1.53133398e-01]
 [ 1.05795500e-01]
 [ 1.88330394e-01]
 [ 9.09583101e-02]
 [ 1.78767023e-01]
 [ 6.51400112e-02]
 [ 6.92596143e-02]
 [ 6.19723003e-02]
 [ 6.17570066e-02]
 [ 1.57064578e-01]
 [ 1.93645638e-01]
 [ 1.94120543e-01]
 [ 1.68963746e-01]
 [ 8.02108387e-02]
 [ 1.85739418e-01]
 [ 1.71155419e-01]
 [ 1.22751106e-01]
 [ 1.21913820e-01]
 [ 9.52089218e-02]
 [ 1.86534826e-01]
 [ 1.44056997e-01]
 [ 1.28013525e-01]
 [ 3.06266291e-02]
 [ 1.33123294e-01]
 [ 1.46984532e-01]
 [-7.38186369e-03]
 [ 1.55018000e-01]
 [ 9.78090471e-02]
 [ 1.59034598e-01]
 [ 1.40113551e-01]
 [ 9.06579315e-02]
 [ 1.00541380e-01]
 [ 5.61213472e-02]
 [ 1.33046297e-01]
 [ 1.77116423e-01]
 [ 1.92512343e-01]
 [ 1.82366701e-01]
 [ 4

tf.Tensor(
[[ 0.1482352 ]
 [ 0.16693034]
 [ 0.0053941 ]
 ...
 [ 0.12596441]
 [-0.01377847]
 [ 0.11564667]], shape=(10100, 1), dtype=float64)
<tf.Variable 'w4:0' shape=(15, 15) dtype=float64, numpy=
array([[-0.34100144,  0.41602031, -0.16343246, -0.1808362 ,  0.04807171,
        -0.16053261, -0.52159709, -0.31433474,  0.38074257, -0.47700208,
         0.40987476,  0.3967891 , -0.00418127, -0.11549504, -0.25131096],
       [-0.34295733, -0.01908323,  0.35596352,  0.55381245,  0.41783095,
         0.32584423, -0.07609645, -0.20105531,  0.55896511,  0.40909973,
        -0.03007801,  0.12477449, -0.02216741, -0.03301425, -0.11319243],
       [-0.00601059,  0.17605726, -0.30755188,  0.2098602 ,  0.05022304,
         0.09995561, -0.18849815, -0.0456722 ,  0.15722377,  0.12420569,
        -0.31109322,  0.01180702,  0.01743309, -0.010612  ,  0.1351424 ],
       [-0.30411157,  0.28108292, -0.00678999,  0.0153868 , -0.06013447,
        -0.25086852,  0.08563361, -0.04351746, -0.49739666, -0.192064

tf.Tensor(
[[ 0.06402694]
 [ 0.06201723]
 [-0.03225173]
 ...
 [ 0.0332263 ]
 [-0.04029403]
 [ 0.02633532]], shape=(10100, 1), dtype=float64)
<tf.Variable 'w4:0' shape=(15, 15) dtype=float64, numpy=
array([[-0.33940405,  0.41590901, -0.15871968, -0.18039408,  0.04923189,
        -0.16101565, -0.52447511, -0.31273298,  0.38247066, -0.4756478 ,
         0.40713268,  0.39616238, -0.00196395, -0.11747219, -0.25103069],
       [-0.34272779, -0.01909564,  0.35663759,  0.55387223,  0.41796239,
         0.32578664, -0.07644211, -0.20085748,  0.55921281,  0.40924899,
        -0.03039376,  0.12468777, -0.0218892 , -0.03325044, -0.11315804],
       [-0.00502844,  0.1759948 , -0.30469069,  0.21012086,  0.05085711,
         0.0996867 , -0.19027408, -0.04468365,  0.158434  ,  0.12496744,
        -0.31260525,  0.01136689,  0.01867943, -0.01172711,  0.1353002 ],
       [-0.30404532,  0.28108054, -0.00661377,  0.01540005, -0.06011962,
        -0.25087669,  0.08549171, -0.04343986, -0.49724338, -0.192033

iteration: 1 lambda2: -2.999300453759221 error: 0.9708906142816669 loss_value: 0.4750668200237147
tf.Tensor(
[[ 0.04297207]
 [ 0.00333449]
 [ 0.04165309]
 [-0.01772573]
 [ 0.04113458]
 [-0.0159991 ]
 [ 0.03682571]
 [ 0.01890369]
 [ 0.06512409]
 [-0.00889662]
 [ 0.0564058 ]
 [-0.01909952]
 [ 0.00511957]
 [-0.01638181]
 [ 0.01258641]
 [ 0.03390495]
 [ 0.07369139]
 [ 0.06504758]
 [ 0.04409349]
 [ 0.00756493]
 [ 0.05202379]
 [ 0.04740452]
 [ 0.01950503]
 [ 0.01537474]
 [ 0.00616778]
 [ 0.0559809 ]
 [ 0.03790693]
 [ 0.01417519]
 [-0.0091278 ]
 [ 0.04289719]
 [ 0.02769943]
 [-0.03781059]
 [ 0.05504113]
 [ 0.01335608]
 [ 0.05174882]
 [ 0.03337532]
 [ 0.00902557]
 [ 0.01042239]
 [-0.00272887]
 [ 0.03099754]
 [ 0.06493211]
 [ 0.05807737]
 [ 0.04759763]
 [-0.00692262]
 [ 0.05048779]
 [-0.00416871]
 [ 0.00747087]
 [ 0.05073271]
 [-0.02319312]
 [-0.01093566]
 [-0.02558095]
 [-0.02774597]
 [ 0.00200997]
 [-0.00148488]
 [-0.02384086]
 [-0.02462212]
 [ 0.06631966]
 [ 0.02980144]
 [ 0.0256276 ]
 [ 0.0

tf.Tensor(
[[ 0.04582975]
 [ 0.03593762]
 [-0.04391327]
 ...
 [ 0.00856874]
 [-0.04881185]
 [ 0.00221533]], shape=(10100, 1), dtype=float64)
<tf.Variable 'w4:0' shape=(15, 15) dtype=float64, numpy=
array([[-0.338727  ,  0.4158837 , -0.15683929, -0.18020077,  0.04945158,
        -0.16112408, -0.52539191, -0.31225489,  0.38343961, -0.47543555,
         0.40659326,  0.39592843, -0.00138209, -0.11801069, -0.25096617],
       [-0.3426614 , -0.01908555,  0.35676742,  0.55388129,  0.41788191,
         0.32580958, -0.07638214, -0.2008876 ,  0.5593464 ,  0.40914191,
        -0.03021858,  0.12466614, -0.0219614 , -0.03317376, -0.11316666],
       [-0.0048535 ,  0.17601929, -0.30433857,  0.21014303,  0.05064921,
         0.09974694, -0.19023519, -0.0447014 ,  0.15885675,  0.12471066,
        -0.31215025,  0.01127161,  0.01847546, -0.01152906,  0.13527626],
       [-0.30415274,  0.2810972 , -0.00696651,  0.01535775, -0.06026376,
        -0.25082312,  0.08574437, -0.04357059, -0.49731969, -0.192194

...

tf.Tensor(
[[ 0.10520689]
 [ 0.02881526]
 [ 0.0911005 ]
 [-0.04917554]
 [ 0.08466382]
 [-0.08697237]
 [ 0.07907639]
 [ 0.04322429]
 [ 0.16940525]
 [-0.06627114]
 [ 0.14131234]
 [-0.0862301 ]
 [ 0.01727819]
 [-0.0704531 ]
 [ 0.05599151]
 [ 0.06359415]
 [ 0.19481286]
 [ 0.17168456]
 [ 0.09885157]
 [ 0.01794104]
 [ 0.12697092]
 [ 0.11084625]
 [ 0.03128329]
 [ 0.01387548]
 [-0.00124928]
 [ 0.14049756]
 [ 0.08959258]
 [ 0.00162625]
 [-0.00783354]
 [ 0.11514095]
 [ 0.04475268]
 [-0.10230602]
 [ 0.14473622]
 [ 0.02703953]
 [ 0.13171111]
 [ 0.07459165]
 [ 0.01503208]
 [ 0.01228399]
 [-0.00446556]
 [ 0.07037861]
 [ 0.16890878]
 [ 0.151079  ]
 [ 0.10986783]
 [-0.01455226]
 [ 0.12087789]
 [-0.05199318]
 [-0.01789693]
 [ 0.13830132]
 [-0.07217701]
 [-0.06086885]
 [-0.05676084]
 [-0.09910723]
 [-0.00591804]
 [-0.00536076]
 [-0.04081828]
 [-0.09594772]
 [ 0.17349147]
 [ 0.05587474]
 [ 0.03610683]
 [ 0.09126898]
 [-0.03763327]
 [ 0.14091566]
 [ 0.0536335 ]
 [ 0.14746465]
 [ 0.02915961]
 [ 0.11619266]

tf.Tensor(
[[ 0.02873655]
 [ 0.1065924 ]
 [-0.07657074]
 [ 0.20111714]
 [ 0.18533556]
 [-0.09931603]
 [ 0.17645088]
 [ 0.05678819]
 [-0.17976464]
 [ 0.19464319]
 [-0.11969315]
 [-0.13084932]
 [-0.21059902]
 [-0.17204025]
 [-0.02817989]
 [ 0.07778447]
 [-0.19541787]
 [ 0.20166485]
 [ 0.20221358]
 [ 0.1515625 ]
 [-0.10539084]
 [ 0.03176128]
 [ 0.17888825]
 [ 0.21925566]
 [ 0.16774301]
 [-0.00713371]
 [ 0.17209442]
 [-0.09015567]
 [ 0.19096444]
 [ 0.18685303]
 [-0.07318618]
 [-0.20562028]
 [-0.02002766]
 [ 0.16919547]
 [ 0.18787184]
 [-0.07491822]
 [ 0.17548024]
 [ 0.18433074]
 [ 0.0133771 ]
 [-0.15505588]
 [ 0.17791119]
 [-0.16525848]
 [-0.14255723]
 [-0.04633688]
 [ 0.03025094]
 [-0.06980704]
 [-0.1640329 ]
 [ 0.19570814]
 [ 0.15675905]
 [-0.11063155]
 [-0.16559888]
 [ 0.10908996]
 [-0.16716741]
 [ 0.08185463]
 [ 0.11766934]
 [-0.04467476]
 [ 0.07641802]
 [-0.17034647]
 [-0.11576096]
 [-0.11405091]
 [ 0.2110722 ]
 [ 0.13856524]
 [-0.13958387]
 [-0.12773415]
 [ 0.08588065]
 [ 0.17402755]

tf.Tensor(
[[ 0.11252038]
 [ 0.07710452]
 [-0.16410712]
 ...
 [-0.02023155]
 [-0.17453765]
 [-0.04040706]], shape=(10100, 1), dtype=float64)
<tf.Variable 'w4:0' shape=(15, 15) dtype=float64, numpy=
array([[-0.3221136 ,  0.41793075, -0.11467464, -0.17559177,  0.04240773,
        -0.15765135, -0.53068185, -0.3091282 ,  0.41430071, -0.48575283,
         0.42586551,  0.38803828, -0.00630526, -0.11328527, -0.25217661],
       [-0.34691066, -0.0174681 ,  0.34207233,  0.55293861,  0.40831943,
         0.32998022, -0.05855972, -0.21024721,  0.55724518,  0.3972473 ,
        -0.0070681 ,  0.12579566, -0.03743607, -0.019024  , -0.11526027],
       [-0.01138778,  0.17953348, -0.32730571,  0.20770652,  0.02885939,
         0.10876113, -0.15911087, -0.05970568,  0.1603444 ,  0.09873453,
        -0.25928101,  0.00991032, -0.01617015,  0.01996923,  0.13080988],
       [-0.30758047,  0.28161709, -0.01748456,  0.01353328, -0.06544521,
        -0.24915501,  0.09325588, -0.04675866, -0.50029719, -0.197739

tf.Tensor(
[[ 0.0904877 ]
 [ 0.00430054]
 [ 0.0845382 ]
 [-0.06968342]
 [ 0.08639831]
 [-0.09901666]
 [ 0.06890997]
 [ 0.02628538]
 [ 0.16232589]
 [-0.07774126]
 [ 0.1335272 ]
 [-0.09985828]
 [-0.00291366]
 [-0.08520236]
 [ 0.03065376]
 [ 0.05736694]
 [ 0.1852936 ]
 [ 0.17079328]
 [ 0.09333645]
 [-0.00060784]
 [ 0.13210507]
 [ 0.1043492 ]
 [ 0.01839043]
 [ 0.00229297]
 [-0.01592038]
 [ 0.14084798]
 [ 0.07531978]
 [-0.00770246]
 [-0.03199903]
 [ 0.09503183]
 [ 0.03671902]
 [-0.12700325]
 [ 0.12596771]
 [ 0.01036866]
 [ 0.11617696]
 [ 0.06108326]
 [-0.00165473]
 [-0.00267413]
 [-0.02474244]
 [ 0.05569588]
 [ 0.15446847]
 [ 0.1581405 ]
 [ 0.11697545]
 [-0.03538525]
 [ 0.12539743]
 [-0.06316172]
 [-0.02805996]
 [ 0.11651248]
 [-0.09144734]
 [-0.07416977]
 [-0.08183626]
 [-0.11581117]
 [-0.02235771]
 [-0.02447823]
 [-0.06820933]
 [-0.111144  ]
 [ 0.16737674]
 [ 0.04585504]
 [ 0.02877218]
 [ 0.09161241]
 [-0.05945545]
 [ 0.12297522]
 [ 0.02883325]
 [ 0.15748205]
 [ 0.02384091]
 [ 0.10360492]

<tf.Variable 'w4:0' shape=(15, 15) dtype=float64, numpy=
array([[-0.32149494,  0.4183276 , -0.11322226, -0.17559404,  0.04103221,
        -0.1569056 , -0.52756963, -0.31088352,  0.41304315, -0.48760724,
         0.42947504,  0.38803172, -0.00765785, -0.11133209, -0.25253311],
       [-0.34819515, -0.01723976,  0.33891502,  0.55295882,  0.40728258,
         0.33062833, -0.05594954, -0.21177293,  0.55615618,  0.39595242,
        -0.00424466,  0.12613162, -0.0398255 , -0.01718199, -0.11565336],
       [-0.00936658,  0.1797493 , -0.32451966,  0.20700212,  0.02757199,
         0.10898731, -0.15803657, -0.05968404,  0.16030047,  0.09699789,
        -0.2564836 ,  0.00935262, -0.01614247,  0.02150392,  0.13097069],
       [-0.30551611,  0.28152473, -0.01409472,  0.01317039, -0.06526638,
        -0.24952078,  0.09160022, -0.04551175, -0.49911317, -0.19762553,
        -0.01859842,  0.10190566, -0.28327576,  0.47108038, -0.51556489],
       [ 0.16462417, -0.12998675, -0.07976581, -0.04027477, -0.

In [ ]:
def solutionplot(u_pred, X_u_train, u_train, X_labelled, u_labelled):
    
    fig, ax = plt.subplots()
    ax.axis('off')

    gs0 = gridspec.GridSpec(1, 4)
    gs0.update(top=1-0.06, bottom=1-1/3, left=0.15, right=0.85, wspace=1)
    ax = plt.subplot(gs0[:, :])

    h = ax.imshow(u_pred, interpolation='nearest', cmap='rainbow', 
                extent=[T.min(), T.max(), X.min(), X.max()], 
                origin='lower', aspect='auto')
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.05)
    fig.colorbar(h, cax=cax)
    
    ax.plot(X_u_train[:,1], X_u_train[:,0], 'kx', label = f'Boundary data ({u_train.shape[0]} points)', markersize = 4, clip_on = False)
    ax.plot(X_labelled[:,1], X_labelled[:,0], 'o', color ='black', label = f'Labelled data ({u_labelled.shape[0]} points)', markersize = 0.1, clip_on = False)

    line = np.linspace(x.min(), x.max(), 2)[:,None]
    ax.plot(t[25]*np.ones((2,1)), line, 'w-', linewidth = 2)
    ax.plot(t[50]*np.ones((2,1)), line, 'w-', linewidth = 2)
    ax.plot(t[75]*np.ones((2,1)), line, 'w-', linewidth = 2)    

    ax.set_xlabel('$t$')
    ax.set_ylabel('$x$')
    ax.legend(frameon=False, loc = 'best')
    ax.set_title('$u(x,t)$', fontsize = 10)
    
    ''' 
    Slices of the solution at points t = 0.25, t = 0.50 and t = 0.75
    '''
    
    ####### Row 1: u(t,x) slices ##################
    gs1 = gridspec.GridSpec(1, 3)
    gs1.update(top=1-1/3, bottom=0, left=0.1, right=0.9, wspace=0.5)

    ax = plt.subplot(gs1[0, 0])
    ax.plot(x,usol.T[25,:], 'b-', linewidth = 2, label = 'Exact')       
    ax.plot(x,u_pred.T[25,:], 'r--', linewidth = 2, label = 'Prediction')
    ax.set_xlabel('$x$')
    ax.set_ylabel('$u(x,t)$')    
    ax.set_title('$t = 0.25s$', fontsize = 10)
    ax.axis('square')
    ax.set_xlim([-1.1,1.1])
    ax.set_ylim([-1.1,1.1])

    ax = plt.subplot(gs1[0, 1])
    ax.plot(x,usol.T[50,:], 'b-', linewidth = 2, label = 'Exact')       
    ax.plot(x,u_pred.T[50,:], 'r--', linewidth = 2, label = 'Prediction')
    ax.set_xlabel('$x$')
    ax.set_ylabel('$u(x,t)$')
    ax.axis('square')
    ax.set_xlim([-1.1,1.1])
    ax.set_ylim([-1.1,1.1])
    ax.set_title('$t = 0.50s$', fontsize = 10)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.35), ncol=5, frameon=False)

    ax = plt.subplot(gs1[0, 2])
    ax.plot(x,usol.T[75,:], 'b-', linewidth = 2, label = 'Exact')       
    ax.plot(x,u_pred.T[75,:], 'r--', linewidth = 2, label = 'Prediction')
    ax.set_xlabel('$x$')
    ax.set_ylabel('$u(x,t)$')
    ax.axis('square')
    ax.set_xlim([-1.1,1.1])
    ax.set_ylim([-1.1,1.1])    
    ax.set_title('$t = 0.75s$', fontsize = 10)
    plt.savefig(os.path.join(newFolder, 'Burgers.png'), dpi = 500)

In [ ]:
print(results)

PINN.set_weights(results.x)

''' Model Accuracy ''' 
u_pred = PINN.forward(X_u_test)

error_vec = np.linalg.norm((u-u_pred),2)/np.linalg.norm(u,2)        # Relative L2 Norm of the error (Vector)
print('Test Error: %.5f'  % (error_vec))

u_pred = np.reshape(u_pred, (256,100), order='F')                        # column-major order

''' Solution Plot '''
solutionplot(u_pred, X_u_train, u_train, X_labelled, u_labelled)

# Plot of collocation points

In [ ]:


# Training data
# X_f_train, X_u_train, u_train , X_labelled, u_labelled = trainingdata(N_b, N_f, N_l)

fig, ax = plt.subplots()

plt.plot(X_u_train[:,1], X_u_train[:,0], '*', color = 'green', markersize = 5, label = f'Boundary Points = {N_b}')
plt.plot(X_f_train[:,1], X_f_train[:,0], 'o', markersize = 0.5, label = f'PDE Residuals = {N_f}')
plt.plot(X_labelled[:,1], X_labelled[:,0], 'P', color = 'red', markersize = 1, label = f'PDE Labelled = {N_l}')
plt.xlabel('t')
plt.ylabel('x')
plt.title('Collocation points')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.savefig(os.path.join(newFolder, 'collocation_points_Burgers.png'), dpi = 500, bbox_inches='tight')
plt.show()


In [ ]:
# PINN.losss_history
fig, ax = plt.subplots()  
ax.plot(PINN.losss_history, 'b')
# ax.set_ylim(0, 5)
ax.set_xlabel('iterations', 
               fontweight ='bold')
ax.set_ylabel('loss', 
               fontweight ='bold')
# ax.grid(True)
  
ax.set_title('evolution of loss\n', fontsize = 14, fontweight ='bold')
plt.legend(['training loss'])
plt.savefig(os.path.join(newFolder, 'loss.png'))
plt.show()

In [ ]:
fig, ax = plt.subplots()  
ax.plot(PINN.error_history, 'r')
# ax.set_xlim(5, 0)
ax.set_xlabel('iterations', 
               fontweight ='bold')
ax.set_ylabel('error', 
               fontweight ='bold')
# ax.grid(True)
  
ax.set_title('evolution error', fontsize = 14, fontweight ='bold')
plt.legend(['error'])
plt.savefig(os.path.join(newFolder, 'error history.png'))
plt.show()

In [ ]:
fig, ax = plt.subplots()  
ax.plot(PINN.w_history)
ax.set_xlabel('iterations', 
               fontweight ='bold')
ax.set_ylabel('w_history', 
               fontweight ='bold')
# ax.grid(True)
  
ax.set_title('training history of the first weight in the NN\n', fontsize = 14, fontweight ='bold')
plt.savefig(os.path.join(newFolder, 'w_history.png'))
plt.show()

In [ ]:
fig, ax = plt.subplots()  
# ax.plot( PINN.lambda2_history, 'b')
ax.plot(PINN.lambda2_history, 'b')
ax.plot(0.5*0.0031830989 * np.ones_like(PINN.lambda2_history), 'r')
# ax.set_ylim(0, 5)
ax.set_xlabel('iterations', 
               fontweight ='bold')
ax.set_ylabel('lamda1', 
               fontweight ='bold')
# ax.grid(True)
  
ax.set_title('evolution of lambda2\n', fontsize = 14, fontweight ='bold')
plt.legend(['predicted', 'exact'])
plt.savefig(os.path.join(newFolder, 'lambda2_history.png'))
plt.show()

In [ ]:
fig, ax = plt.subplots()  
# ax.plot( PINN.lambda2_history, 'b')
ax.plot(range(2000, 8000), PINN.lambda2_history[2000:], 'b')
ax.plot(range(2000, 8000), 0.5*0.0031830989 * np.ones_like(PINN.lambda2_history[2000:]), 'r')
# ax.set_ylim(0, 5)
ax.set_xlabel('iterations', 
               fontweight ='bold')
ax.set_ylabel('lamda1', 
               fontweight ='bold')
# ax.grid(True)
  
ax.set_title('evolution of lambda2\n', fontsize = 14, fontweight ='bold')
plt.legend(['predicted', 'exact'])
plt.savefig(os.path.join(newFolder, 'lambda2_history_magnified.png'))
plt.show()

In [ ]:
print(f'        *RUN INFORMATION*{foldername}')
print('seed:', seed)
print('layers:', layers)
print('N_f:', N_f, 'N_b:', N_b, 'N_l:', N_l)
print('max iteration:', maxiter)
print('')
print('run time:', elapsed/60.0, '(m)')
print(f'lambda2={PINN.lambda2[0].numpy()}')
print('training loss:', PINN.losss_history[-1])
print(f'error: {PINN.error_history[-1]}')


with open(os.path.join(newFolder, foldername+'.txt'), 'w') as file:    
    file.write(f'seed: {seed}\n')
    file.write(f'layers: {layers}\n')
    file.write(f'N_f:, {N_f}, N_b:, {N_b}, N_l:, {N_l}\n')
    file.write(f'max iteration: {maxiter}\n')
    file.write(f'\n')
    file.write(f'run time: {elapsed/60.0}(min)\n')
    file.write(f'lambda2=, {PINN.lambda2[0].numpy()}\n')
    file.write(f'training loss: {PINN.losss_history[-1]}\n')   
    file.write(f'error:, {PINN.error_history[-1]}\n')